# Reinforcement Learning - DQN for Camera Control

In [ ]:
import gym
from gym import spaces
import numpy as np
import tensorflow as tf
from collections import deque
import random
import matplotlib.pyplot as plt

## Define the custom environment

In [ ]:

class CameraControlEnv(gym.Env):
    """
    Custom Gym environment simulating a camera control task.
    The environment returns an observation vector and expects an action that selects the best signal source.
    """
    def __init__(self):
        super(CameraControlEnv, self).__init__()
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(
            low=np.array([0, 0, 0] * 3),
            high=np.array([1, 1, 100] * 3),
            dtype=np.float32
        )

    def reset(self):
        """
        Resets the environment to a random initial state.

        Returns:
            np.ndarray: Initial state of the environment.
        """
        self.state = np.array([
            np.random.randint(0, 2), np.random.randint(0, 2), np.random.randint(1, 101),
            np.random.randint(0, 2), np.random.randint(0, 2), np.random.randint(1, 101),
            np.random.randint(0, 2), np.random.randint(0, 2), np.random.randint(1, 101)
        ], dtype=np.float32)
        return self.state

    def step(self, action):
        """
        Takes an action and returns the new state, reward, and termination flag.

        Args:
            action (int): Index of the action to take.

        Returns:
            tuple: (state, reward, done, info)
        """
        signals = [
            0.7 * (self.state[2] / 100) + 0.15 * self.state[0] + 0.15 * self.state[1],
            0.7 * (self.state[5] / 100) + 0.15 * self.state[3] + 0.15 * self.state[4],
            0.7 * (self.state[8] / 100) + 0.15 * self.state[6] + 0.15 * self.state[7]
        ]
        reward = 5 if action == np.argmax(signals) else -1
        done = True
        return self.state, reward, done, {}
        

## DQN Agent definition

In [ ]:

class DQNAgent:
    """
    Deep Q-Network Agent for reinforcement learning in custom environments.
    Trains a neural network to learn optimal actions from environment state transitions.
    """
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=100000)
        self.gamma = 0.75
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        self.batch_size = 64
        self.model = self._build_model()
        self.losses = []

    def _build_model(self):
        """
        Builds a neural network model for the DQN agent.

        Returns:
            keras.Model: Compiled neural network model.
        """
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(24, input_shape=(self.state_size,), activation='relu'),
            tf.keras.layers.Dense(24, activation='relu'),
            tf.keras.layers.Dense(self.action_size, activation='linear')
        ])
        model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate=self.learning_rate))
        return model

    def remember(self, state, action, reward, next_state, done):
        """
        remember method.
        """
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        """
        act method.
        """
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        act_values = self.model.predict(state, verbose=0)
        return np.argmax(act_values[0])

    def replay(self):
        """
        replay method.
        """
        if len(self.memory) < self.batch_size:
            return
        minibatch = random.sample(self.memory, self.batch_size)
        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                target = reward + self.gamma * np.amax(self.model.predict(next_state, verbose=0)[0])
            target_f = self.model.predict(state, verbose=0)
            target_f[0][action] = target
            history = self.model.fit(state, target_f, epochs=1, verbose=0)
            loss = history.history['loss'][0]
            self.losses.append(loss)

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        